# 04 Maßnahmenableitung

### 4.1 Maßnahmenkatalog als Stammdaten

Der Maßnahmenkatalog (`katalog`) ist das Regelwerk des Smart Service. Er funktioniert wie Stammdaten (Master Data): Er wird einmal fachlich festgelegt, ändert sich selten und wird bei jeder Auswertung unverändert verwendet. Neue Maßnahmen können ergänzt werden, ohne den Programmcode zu ändern.

Jeder Eintrag ist eine **Wenn-dann-Regel**:

| Feld | Bedeutung |
|---|---|
| `ID` | Kennung der Maßnahme (M1–M7) |
| `Situation` | Beschreibung der Problemsituation |
| `Bedingung` | **WENN:** woran die Situation in den Daten erkannt wird |
| `Massnahme` | **DANN:** empfohlene Maßnahme |
| `Ebene` | Auswertung je Unfall oder je Fahrzeug |
| `3E`, `Quelle` | Art der Maßnahme und fachliche Grundlage |

**Regeln:**

| ID | WENN … | DANN … |
|---|---|---|
| M1 | Unfall abends/nachts (20–5 Uhr) in einem Wohn- oder Bürogebiet | Abendliche Geschwindigkeitskontrollen, Tempo 30, Verkehrsberuhigung |
| M2 | Kollision mit einem Objekt am Straßenrand | Schutzplanken, Leitpfosten, Absicherung von Hindernissen |
| M3 | Fahrer jünger als 18 Jahre | Führerscheinkontrollen, Halterhaftung |
| M4 | Ursache Überholen oder Fahren auf der Gegenspur | Überholverbote, Mitteltrennung |
| M5 | Dunkelheit ohne Straßenbeleuchtung | Straßenbeleuchtung nachrüsten |
| M6 | Fußgängerunfall bei Dunkelheit | Beleuchtete Fußgängerüberwege, Mittelinseln |
| M7 | Unfall auf freier Strecke abends/nachts | Streckenbezogene Geschwindigkeitsüberwachung, Leitpfosten, Markierung |

### 4.2 Anwendung der Regeln

Die Funktion `massnahmen_empfehlen()` wendet den Katalog auf einen beliebigen Datenausschnitt an – z. B. die ganze Stadt oder eine einzelne Hotspot-Zelle:

1. Für jede Regel werden die Unfälle ausgewählt, auf die die **Bedingung** zutrifft.
2. Regeln mit zu wenigen passenden Unfällen werden übersprungen.
3. Für die übrigen Regeln wird der **Überschuss schwerer Unfälle** berechnet.
4. Regeln **ohne Überschuss** entfallen – dort ist die Situation nicht gefährlicher als der Durchschnitt.
5. Die verbleibenden Maßnahmen werden **nach Überschuss sortiert** ausgegeben.

Beispiel: Anwendung auf Hotspot-Zelle: HS2 – 122 Unfälle, davon 28 schwer

| Regel | passende Unfälle | davon schwer | Anteil | erwartet (n × 16,6 %) | Überschuss |
|---|---|---|---|---|---|
| **M1** Abend/Nacht in Büro-/Wohngebiet | 21 | 9 | 42,9 % | 3,5 | **+6** |
| M2 Objekt am Straßenrand | 16 | 3 | 18,8 % | 2,7 | 0 |
| übrige Regeln | – | – | – | – | zu wenige Fälle oder kein Überschuss |

→ Empfehlung HS2: M1 – abendliche Geschwindigkeitskontrollen, Tempo 30

### 4.3 Kombination von Maßnahmen

In einem Gebiet können mehrere Regeln gleichzeitig zutreffen. In HS2 sind beispielsweise sowohl nächtliche Unfälle in Wohn- und Bürogebieten (M1) als auch Kollisionen mit Objekten am Straßenrand (M2) auffällig. Die Funktion `massnahmen_empfehlen()` gibt in diesem Fall eine **nach Überschuss priorisierte Liste** aus. Die Verkehrsbehörde kann daraufhin

- nur die **erste Maßnahme** umsetzen (größter Überschuss, geringste Kosten), oder
- mehrere Maßnahmen zu einem **Maßnahmenpaket** bündeln, wenn das Budget es zulässt.

Die Wirkungen mehrerer Maßnahmen **addieren sich nicht**, da sie teilweise dieselben Unfälle betreffen: Ein Unfall, der durch die erste Maßnahme bereits verhindert wird, kann durch die zweite nicht ein weiteres Mal verhindert werden. Üblich ist die multiplikative Kombination der *Crash Modification Factors* (CMF = 1 − Wirkung), wie sie im *Highway Safety Manual* (AASHTO 2010) und von iRAP verwendet wird:

> Gesamtwirkung = 1 − (1 − w₁) · (1 − w₂)
>
> Beispiel: zwei Maßnahmen mit je 30 % Wirkung → 1 − 0,7 · 0,7 = **51 %** (nicht 60 %)

Diese Methode setzt voraus, dass die Maßnahmen unabhängig voneinander wirken. Richten sich mehrere Maßnahmen auf dieselbe Unfallart – etwa M1 und M7, die beide nächtliche Unfälle betreffen –, kann auch sie die Wirkung noch überschätzen. Es wird daher empfohlen, höchstens drei Maßnahmen auf diese Weise zu kombinieren (Gross & Hamidi 2011).

Im Prototyp wird je Hotspot nur die Maßnahme mit dem größten Überschuss umgesetzt und in der Simulation bewertet. Die Zusammenstellung optimaler Maßnahmenpakete innerhalb eines Budgets ist eine Erweiterung für den Betrieb.


### 4.4 Erkennung neuer Problemsituationen

Der Katalog erkennt nur Situationen, die vorab als Regel definiert sind. Deshalb sucht ein Entscheidungsbaum ohne Vorgaben nach Merkmalskombinationen mit erhöhtem Anteil schwerer Unfälle und prüft, ob sie bereits durch eine Regel abgedeckt sind (Kapitel Modell).

Er findet im Wesentlichen dieselben Situationen wie die manuelle Analyse (nachts auf freier Strecke → M7, Objekte am Straßenrand → M2) und bestätigt damit den Katalog. Im Betrieb läuft diese Prüfung regelmäßig; neue, nicht abgedeckte Situationen werden Fachleuten gemeldet, die eine Maßnahme zuordnen und den Katalog ergänzen. So bleibt der Service lernfähig.

**QUELLE**
- AASHTO (2010): *Highway Safety Manual*, 1st Edition. Washington, D.C.: American Association of State Highway and Transportation Officials.
- Gross, F.; Hamidi, A. (2011): *Investigation of Existing and Alternative Methods for Combining Multiple CMFs*. Federal Highway Administration. https://cmfclearinghouse.fhwa.dot.gov/collateral/Combining_Multiple_CMFs_Final.pdf


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_clean  = pd.read_parquet('../../data/df_features.parquet')
df_unfall = pd.read_parquet('../../data/df_unfall.parquet')


In [10]:
from katalog import katalog
from katalog import massnahmen_empfehlen

In [11]:
empfehlung = massnahmen_empfehlen(df_unfall, df_clean, katalog)
empfehlung

,ID,Situation,Ebene,n,Anteil_schwer_%,Ueberschuss,Massnahme,3E,Quelle
Prio,,,,,,,,,
1,M3,Fahrer unter 18 Jahren,Fahrzeug,825,21.9,54,"Führerscheinkontrollen, Halterhaftung",Enforcement,WHO Save LIVES: Enforcement
2,M7,Freie Strecke (kein Knotenpunkt) abends/nachts,Unfall,584,25.7,53,"Streckenbezogene Geschwindigkeitsüberwachung, ...","Enforcement, Engineering",WHO Save LIVES: Speed management; iRAP: Deline...
3,M1,Abend/Nacht (20–5 Uhr) in Büro-/Wohngebieten,Unfall,817,22.0,45,"Abendliche Geschwindigkeitskontrollen, Tempo 3...","Enforcement, Engineering",WHO Save LIVES: Speed management
4,M4,Überholen / Fahren auf der Gegenspur,Fahrzeug,714,19.0,26,"Überholverbote, Mittelleitplanken","Engineering, Enforcement",iRAP: Median
5,M2,Kollision mit Objekten am Straßenrand,Unfall,1373,17.8,17,"Schutzplanken, Leitpfosten, Absicherung von Hi...",Engineering,iRAP: Roadside safety barriers
6,M5,Dunkelheit ohne Straßenbeleuchtung,Unfall,157,26.1,15,Straßenbeleuchtung nachrüsten,Engineering,iRAP: Street Lighting
7,M6,Fußgängerunfälle bei Dunkelheit,Unfall,219,22.8,14,"Beleuchtete Fußgängerüberwege, Mittelinseln",Engineering,iRAP: Pedestrian Crossing Facilities


### Maßnahmenkatalog

| Prio | Situation | Anteil schwer | Überschuss | Maßnahme | 3E |
|---|---|---|---|---|---|
| 1 | M3 – Fahrer unter 18 Jahren | 21,9 % | +54 | Führerscheinkontrollen, Halterhaftung | Enforcement |
| 2 | M7 – Freie Strecke abends/nachts | 25,6 % | +53 | Streckenbezogene Geschwindigkeitsüberwachung, Leitpfosten, Markierung | Enforcement, Engineering |
| 3 | M1 – Abend/Nacht in Büro-/Wohngebieten | 22,0 % | +45 | Abendliche Geschwindigkeitskontrollen, Tempo 30, Verkehrsberuhigung | Enforcement, Engineering |
| 4 | M4 – Überholen / Fahren auf der Gegenspur | 19,0 % | +26 | Überholverbote, Mittelleitplanken | Engineering, Enforcement |
| 5 | M2 – Kollision mit Objekten am Straßenrand | 18,2 % | +23 | Schutzplanken, Leitpfosten, Absicherung von Hindernissen | Engineering |
| 6 | M6 – Fußgängerunfälle bei Dunkelheit | 24,5 % | +17 | Beleuchtete Fußgängerüberwege, Mittelinseln | Engineering |
| 7 | M5 – Dunkelheit ohne Straßenbeleuchtung | 26,1 % | +15 | Straßenbeleuchtung nachrüsten | Engineering |


**M1 – Geschwindigkeitsmanagement in Büro- und Wohngebieten am Abend und in der Nacht**
- *Problem:* 22,0 % der Unfälle zwischen 20 und 5 Uhr in Büro- und Wohngebieten sind schwer (n = 817, +45); in Wohngebieten am Abend 23,1 %.
- *Maßnahme:* mobile Geschwindigkeitskontrollen zwischen 20 und 5 Uhr, Tempo 30 in Wohngebieten, bauliche Verkehrsberuhigung.
- *Begründung:* Bei geringem Verkehr abends und nachts wird schneller gefahren; geringere Geschwindigkeit senkt Unfallrisiko und -schwere.
- *Quelle:* WHO Save LIVES – Speed management; iRAP – Speed Management / Traffic Calming

**M2 – Schutzeinrichtungen am Straßenrand**
- *Problem:* Kollisionen mit Objekten am Straßenrand sind zu 18,2 % schwer (n = 1.372, +23).
- *Maßnahme:* Schutzplanken an Gefahrenstellen, Leitpfosten und Markierungen, Entfernen oder Absichern von Hindernissen.
- *Begründung:* Kommt ein Fahrzeug von der Fahrbahn ab, verringern Schutzeinrichtungen die Aufprallschwere.
- *Quelle:* WHO Save LIVES – Infrastructure design; iRAP – Roadside safety barriers

**M3 – Führerscheinkontrollen bei jungen Fahrern**
- *Problem:* Fahrer unter 18 Jahren: 21,9 % schwer (n = 825, +54).
- *Maßnahme:* verstärkte Kontrollen der Fahrerlaubnis, Halterhaftung bei Überlassung an Minderjährige.
- *Begründung:* Fahrer unter 18 Jahren besitzen in der Regel keine gültige Fahrerlaubnis und keine geprüfte Fahrkompetenz.
- *Quelle:* WHO Save LIVES – Enforcement of traffic laws

**M4 – Maßnahmen gegen gefährliches Überholen**
- *Problem:* Überholen / Fahren auf der Gegenspur: 19,0 % schwer (n = 714, +26).
- *Maßnahme:* Überholverbote an unübersichtlichen Stellen, bauliche Mitteltrennung.
- *Begründung:* Überholen und Fahren auf der Gegenspur führen zu Frontalkollisionen mit hoher Aufprallenergie.
- *Quelle:* WHO Save LIVES – Infrastructure design; iRAP – Median

**M5 – Straßenbeleuchtung nachrüsten**
- *Problem:* Dunkelheit ohne Beleuchtung: 26,1 % schwer (n = 157, +15).
- *Maßnahme:* Installation bzw. Instandsetzung von Straßenbeleuchtung in Hotspot-Zellen.
- *Begründung:* Ohne Beleuchtung werden Hindernisse und andere Verkehrsteilnehmer zu spät erkannt.
- *Quelle:* iRAP – Street Lighting

**M6 – Sichere Fußgängerquerungen bei Dunkelheit**
- *Problem:* Fußgängerunfälle bei Dunkelheit: 24,5 % schwer (n = 212, +17); abends und nachts sogar 32–33 %.
- *Maßnahme:* beleuchtete Fußgängerüberwege, Mittelinseln, reflektierende Markierungen.
- *Begründung:* Fußgänger werden bei Dunkelheit schlecht gesehen und sind bei einem Zusammenstoß ungeschützt.
- *Quelle:* iRAP – Pedestrian Crossing Facilities

**M7 – Geschwindigkeit und Führung auf freier Strecke abends/nachts**
- *Problem:* Unfälle außerhalb von Knotenpunkten zwischen 20 und 5 Uhr: 25,6 % schwer (n = 586, +53).
- *Maßnahme:* streckenbezogene Geschwindigkeitsüberwachung, Leitpfosten und Fahrbahnmarkierungen für bessere Erkennbarkeit des Straßenverlaufs.
- *Begründung:* Auf freier Strecke wird nachts schneller gefahren und der Straßenverlauf schlechter erkannt.
- *Quelle:* WHO Save LIVES – Speed management; iRAP – Delineation

QUELLE:
1. WHO Save LIVES – A Road Safety Technical Package (2017) https://www.who.int/publications/i/item/save-lives-a-road-safety-technical-package
2. iRAP Road Safety Toolkit https://toolkit.irap.org/
